# Week 10 — Wednesday Lab Notebook  
## Improving Training

**Topic focus:** overfitting, regularization, dropout, weight decay, early stopping, monitoring loss and metrics, saving/loading checkpoints

This notebook is written in simple language for lab teaching.  
We will use **PyTorch** and a small classification dataset so students can focus on training behavior.


## 3-Hour Structure

**Hour 1**
- understand what overfitting means
- build a baseline neural network
- monitor train loss and validation loss

**Hour 2**
- improve training using dropout
- improve training using weight decay
- compare models using plots and scores

**Hour 3**
- understand early stopping
- save the best checkpoint
- load the model again
- discuss common mistakes and best practices


## Part A — Why Training Needs Improvement

Sometimes a model learns the training data very well, but performs worse on new data.

This is called **overfitting**.

Today we will not just train a model.  
We will also learn how to make training **more reliable**.

We will answer these questions:

- how do we know overfitting is happening?
- what is the difference between training loss and validation loss?
- how does dropout help?
- how does weight decay help?
- what is early stopping?
- why do we save the best checkpoint?


## Part B — Import Libraries


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report

import torch
import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader

torch.manual_seed(42)
np.random.seed(42)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)


## Part C — Load and Prepare a Small Dataset

We will use the **Breast Cancer Wisconsin** dataset from scikit-learn.

Why this dataset?

- it is small
- it is already clean
- it is useful for binary classification
- it trains quickly in class

We will split the data into:

- **training set**
- **validation set**
- **test set**

Why three parts?

- training set = model learns here
- validation set = we check model during training
- test set = final honest result


In [ ]:
data = load_breast_cancer()

X = data.data
y = data.target

feature_names = data.feature_names
target_names = data.target_names

print("X shape:", X.shape)
print("y shape:", y.shape)
print("Classes:", target_names)
print("\nFirst 5 feature names:")
print(feature_names[:5])


In [ ]:
# First split: training and temporary
X_train, X_temp, y_train, y_temp = train_test_split(
    X, y, test_size=0.30, random_state=42, stratify=y
)

# Second split: validation and test
X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp, test_size=0.50, random_state=42, stratify=y_temp
)

print("Training shape:", X_train.shape)
print("Validation shape:", X_val.shape)
print("Test shape:", X_test.shape)


### Teaching point

Always fit the scaler on **training data only**.

Do not fit the scaler on the whole dataset.

If you do that, information from validation/test can leak into training.


In [ ]:
scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train)
X_val_scaled = scaler.transform(X_val)
X_test_scaled = scaler.transform(X_test)

X_train_tensor = torch.tensor(X_train_scaled, dtype=torch.float32)
X_val_tensor = torch.tensor(X_val_scaled, dtype=torch.float32)
X_test_tensor = torch.tensor(X_test_scaled, dtype=torch.float32)

y_train_tensor = torch.tensor(y_train, dtype=torch.float32).view(-1, 1)
y_val_tensor = torch.tensor(y_val, dtype=torch.float32).view(-1, 1)
y_test_tensor = torch.tensor(y_test, dtype=torch.float32).view(-1, 1)

train_ds = TensorDataset(X_train_tensor, y_train_tensor)
val_ds = TensorDataset(X_val_tensor, y_val_tensor)
test_ds = TensorDataset(X_test_tensor, y_test_tensor)

train_loader = DataLoader(train_ds, batch_size=32, shuffle=True)
val_loader = DataLoader(val_ds, batch_size=64, shuffle=False)
test_loader = DataLoader(test_ds, batch_size=64, shuffle=False)

print("Train batches:", len(train_loader))
print("Validation batches:", len(val_loader))
print("Test batches:", len(test_loader))


## Part D — A Helper Function for Accuracy

Our model will give **logits**.

For binary classification:
- logits go through sigmoid to become probabilities
- if probability >= 0.5, predict class 1
- otherwise predict class 0


In [ ]:
def binary_accuracy_from_logits(logits, targets):
    probs = torch.sigmoid(logits)
    preds = (probs >= 0.5).float()
    acc = (preds == targets).float().mean().item()
    return acc


## Part E — A General Training Function

This function will:
- train the model
- calculate training loss and accuracy
- calculate validation loss and accuracy
- optionally use **early stopping**
- optionally save the **best checkpoint**


In [ ]:
def train_model(
    model,
    train_loader,
    val_loader,
    criterion,
    optimizer,
    epochs=50,
    patience=None,
    checkpoint_path=None,
    device=device,
):
    history = {
        "train_loss": [],
        "val_loss": [],
        "train_acc": [],
        "val_acc": [],
    }

    best_val_loss = float("inf")
    best_epoch = -1
    patience_counter = 0

    model.to(device)

    for epoch in range(epochs):
        # ---------- training ----------
        model.train()
        running_train_loss = 0.0
        running_train_acc = 0.0
        train_count = 0

        for xb, yb in train_loader:
            xb, yb = xb.to(device), yb.to(device)

            optimizer.zero_grad()
            logits = model(xb)
            loss = criterion(logits, yb)
            loss.backward()
            optimizer.step()

            batch_size = xb.size(0)
            running_train_loss += loss.item() * batch_size
            running_train_acc += binary_accuracy_from_logits(logits, yb) * batch_size
            train_count += batch_size

        epoch_train_loss = running_train_loss / train_count
        epoch_train_acc = running_train_acc / train_count

        # ---------- validation ----------
        model.eval()
        running_val_loss = 0.0
        running_val_acc = 0.0
        val_count = 0

        with torch.no_grad():
            for xb, yb in val_loader:
                xb, yb = xb.to(device), yb.to(device)

                logits = model(xb)
                loss = criterion(logits, yb)

                batch_size = xb.size(0)
                running_val_loss += loss.item() * batch_size
                running_val_acc += binary_accuracy_from_logits(logits, yb) * batch_size
                val_count += batch_size

        epoch_val_loss = running_val_loss / val_count
        epoch_val_acc = running_val_acc / val_count

        history["train_loss"].append(epoch_train_loss)
        history["val_loss"].append(epoch_val_loss)
        history["train_acc"].append(epoch_train_acc)
        history["val_acc"].append(epoch_val_acc)

        print(
            f"Epoch {epoch+1:02d}/{epochs} | "
            f"Train Loss: {epoch_train_loss:.4f} | "
            f"Val Loss: {epoch_val_loss:.4f} | "
            f"Train Acc: {epoch_train_acc:.4f} | "
            f"Val Acc: {epoch_val_acc:.4f}"
        )

        # ---------- early stopping / checkpoint ----------
        if epoch_val_loss < best_val_loss:
            best_val_loss = epoch_val_loss
            best_epoch = epoch + 1
            patience_counter = 0

            if checkpoint_path is not None:
                torch.save(model.state_dict(), checkpoint_path)
        else:
            patience_counter += 1

        if patience is not None and patience_counter >= patience:
            print(f"Early stopping triggered at epoch {epoch+1}.")
            break

    return history, best_val_loss, best_epoch


## Part F — Function to Plot Training History


In [ ]:
def plot_history(history, title_prefix="Model"):
    epochs = range(1, len(history["train_loss"]) + 1)

    plt.figure(figsize=(12, 4))

    plt.subplot(1, 2, 1)
    plt.plot(epochs, history["train_loss"], label="Train Loss")
    plt.plot(epochs, history["val_loss"], label="Validation Loss")
    plt.title(f"{title_prefix} - Loss")
    plt.xlabel("Epoch")
    plt.ylabel("Loss")
    plt.legend()

    plt.subplot(1, 2, 2)
    plt.plot(epochs, history["train_acc"], label="Train Accuracy")
    plt.plot(epochs, history["val_acc"], label="Validation Accuracy")
    plt.title(f"{title_prefix} - Accuracy")
    plt.xlabel("Epoch")
    plt.ylabel("Accuracy")
    plt.legend()

    plt.tight_layout()
    plt.show()


## Part G — Function to Evaluate on Test Data


In [ ]:
def evaluate_model(model, test_loader, device=device):
    model.eval()
    all_preds = []
    all_true = []

    with torch.no_grad():
        for xb, yb in test_loader:
            xb = xb.to(device)
            logits = model(xb)
            probs = torch.sigmoid(logits)
            preds = (probs >= 0.5).int().cpu().numpy().ravel()

            all_preds.extend(preds.tolist())
            all_true.extend(yb.numpy().astype(int).ravel().tolist())

    acc = accuracy_score(all_true, all_preds)
    cm = confusion_matrix(all_true, all_preds)

    return acc, cm, classification_report(all_true, all_preds, target_names=target_names)


## Part H — Baseline Model Without Regularization

First we will build a model that has:
- more hidden units
- no dropout
- no weight decay

This will act as our **baseline**.

If training accuracy becomes very high but validation performance stops improving, that is a sign of overfitting.


In [ ]:
class BaselineNet(nn.Module):
    def __init__(self, input_size):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(input_size, 128),
            nn.ReLU(),
            nn.Linear(128, 64),
            nn.ReLU(),
            nn.Linear(64, 1),
        )

    def forward(self, x):
        return self.net(x)


In [ ]:
input_size = X_train_tensor.shape[1]

baseline_model = BaselineNet(input_size).to(device)
criterion = nn.BCEWithLogitsLoss()
baseline_optimizer = torch.optim.Adam(baseline_model.parameters(), lr=0.001)

baseline_history, baseline_best_val_loss, baseline_best_epoch = train_model(
    baseline_model,
    train_loader,
    val_loader,
    criterion,
    baseline_optimizer,
    epochs=50,
    patience=None,
    checkpoint_path=None,
)


In [ ]:
plot_history(baseline_history, title_prefix="Baseline Model")


In [ ]:
baseline_test_acc, baseline_cm, baseline_report = evaluate_model(baseline_model, test_loader)

print("Baseline test accuracy:", round(baseline_test_acc, 4))
print("\nConfusion Matrix:\n", baseline_cm)
print("\nClassification Report:\n", baseline_report)


### What to observe

Look at:
- training loss
- validation loss
- training accuracy
- validation accuracy

Questions for students:
- Is training accuracy increasing?
- Is validation accuracy also increasing at the same speed?
- Does validation loss stop improving before training loss?


## Part I — What is Overfitting?

**Overfitting** means the model learns the training data too closely.

It may memorize patterns, noise, or very specific details.

Then:
- training performance becomes excellent
- validation/test performance does not improve much
- sometimes validation performance becomes worse

Simple idea:

- **underfitting** = model is too simple
- **good fitting** = model learns useful patterns
- **overfitting** = model learns too much from training details


## Part J — Add Dropout

**Dropout** randomly switches off some neurons during training.

This forces the network not to depend too much on specific neurons.

Because of this:
- the model becomes less dependent on one path
- generalization can improve
- overfitting can reduce


In [ ]:
class DropoutNet(nn.Module):
    def __init__(self, input_size, dropout_rate=0.3):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(input_size, 128),
            nn.ReLU(),
            nn.Dropout(dropout_rate),
            nn.Linear(128, 64),
            nn.ReLU(),
            nn.Dropout(dropout_rate),
            nn.Linear(64, 1),
        )

    def forward(self, x):
        return self.net(x)


In [ ]:
dropout_model = DropoutNet(input_size, dropout_rate=0.3).to(device)
criterion = nn.BCEWithLogitsLoss()
dropout_optimizer = torch.optim.Adam(dropout_model.parameters(), lr=0.001)

dropout_history, dropout_best_val_loss, dropout_best_epoch = train_model(
    dropout_model,
    train_loader,
    val_loader,
    criterion,
    dropout_optimizer,
    epochs=50,
    patience=None,
    checkpoint_path=None,
)


In [ ]:
plot_history(dropout_history, title_prefix="Dropout Model")


In [ ]:
dropout_test_acc, dropout_cm, dropout_report = evaluate_model(dropout_model, test_loader)

print("Dropout test accuracy:", round(dropout_test_acc, 4))
print("\nConfusion Matrix:\n", dropout_cm)
print("\nClassification Report:\n", dropout_report)


### Teaching point

Explain in simple words:

During training, dropout says:

> "Do not trust only one neuron. Learn from multiple paths."

During testing, dropout is turned off automatically.


## Part K — Add Weight Decay

**Weight decay** is another regularization method.

It adds a penalty when model weights become too large.

Simple idea:
- very large weights can make the model too sensitive
- weight decay tries to keep weights smaller
- this can improve generalization


In [ ]:
weight_decay_model = BaselineNet(input_size).to(device)
criterion = nn.BCEWithLogitsLoss()

# Here weight_decay is added in Adam
weight_decay_optimizer = torch.optim.Adam(
    weight_decay_model.parameters(),
    lr=0.001,
    weight_decay=1e-3
)

weight_decay_history, wd_best_val_loss, wd_best_epoch = train_model(
    weight_decay_model,
    train_loader,
    val_loader,
    criterion,
    weight_decay_optimizer,
    epochs=50,
    patience=None,
    checkpoint_path=None,
)


In [ ]:
plot_history(weight_decay_history, title_prefix="Weight Decay Model")


In [ ]:
wd_test_acc, wd_cm, wd_report = evaluate_model(weight_decay_model, test_loader)

print("Weight decay test accuracy:", round(wd_test_acc, 4))
print("\nConfusion Matrix:\n", wd_cm)
print("\nClassification Report:\n", wd_report)


## Part L — Compare the Three Models

Now we compare:
- baseline
- dropout
- weight decay


In [ ]:
comparison_df = pd.DataFrame({
    "Model": ["Baseline", "Dropout", "Weight Decay"],
    "Best Validation Loss": [
        baseline_best_val_loss,
        dropout_best_val_loss,
        wd_best_val_loss
    ],
    "Best Validation Epoch": [
        baseline_best_epoch,
        dropout_best_epoch,
        wd_best_epoch
    ],
    "Final Test Accuracy": [
        baseline_test_acc,
        dropout_test_acc,
        wd_test_acc
    ]
})

comparison_df.round(4)


### Discussion

Ask students:

- Which model gave the lowest validation loss?
- Which model gave the best test accuracy?
- Did the improved model always have the highest training accuracy?
- Why is **validation behavior** more important than only training behavior?


## Part M — Monitoring Loss and Metrics

During training we should monitor more than one thing.

Common items to monitor:
- training loss
- validation loss
- training accuracy
- validation accuracy

Why?

Because one metric alone may hide a problem.

Example:
- training accuracy can look great
- but validation loss may still be increasing


In [ ]:
last_epochs_df = pd.DataFrame({
    "Epoch": range(1, len(baseline_history["train_loss"]) + 1),
    "Train Loss": baseline_history["train_loss"],
    "Val Loss": baseline_history["val_loss"],
    "Train Acc": baseline_history["train_acc"],
    "Val Acc": baseline_history["val_acc"],
})

last_epochs_df.tail(10).round(4)


### Simple explanation

- **loss** tells how wrong the model is
- **accuracy** tells how many predictions are correct
- sometimes accuracy changes slowly
- loss can show warning signs earlier

That is why both are useful.


## Part N — Early Stopping

**Early stopping** means:

> stop training when validation loss stops improving

This helps because:
- we do not waste time
- we reduce overfitting
- we keep the best model from the best validation epoch

We will now train a model with:
- dropout
- weight decay
- early stopping
- checkpoint saving


In [ ]:
checkpoint_path = "best_week10_wednesday_model.pt"

improved_model = DropoutNet(input_size, dropout_rate=0.3).to(device)
criterion = nn.BCEWithLogitsLoss()

improved_optimizer = torch.optim.Adam(
    improved_model.parameters(),
    lr=0.001,
    weight_decay=1e-3
)

improved_history, improved_best_val_loss, improved_best_epoch = train_model(
    improved_model,
    train_loader,
    val_loader,
    criterion,
    improved_optimizer,
    epochs=100,
    patience=10,
    checkpoint_path=checkpoint_path,
)


In [ ]:
plot_history(improved_history, title_prefix="Improved Model with Early Stopping")


### Teaching point

Notice:
- training may stop before the maximum number of epochs
- the best epoch is not always the last epoch
- that is why checkpoint saving matters


## Part O — Load the Best Checkpoint Again


In [ ]:
best_loaded_model = DropoutNet(input_size, dropout_rate=0.3).to(device)
best_loaded_model.load_state_dict(torch.load(checkpoint_path, map_location=device))

best_test_acc, best_cm, best_report = evaluate_model(best_loaded_model, test_loader)

print("Loaded best model test accuracy:", round(best_test_acc, 4))
print("\nConfusion Matrix:\n", best_cm)
print("\nClassification Report:\n", best_report)


In [ ]:
with torch.no_grad():
    sample_logits = best_loaded_model(X_test_tensor[:10].to(device))
    sample_probs = torch.sigmoid(sample_logits).cpu().numpy().ravel()
    sample_preds = (sample_probs >= 0.5).astype(int)

print("Predicted classes:", sample_preds)
print("True classes     :", y_test[:10])
print("Probabilities    :", np.round(sample_probs, 4))


## Part P — Save a Full Checkpoint Dictionary

Sometimes we do not want to save only the model weights.

We may also want to save:
- optimizer state
- epoch number
- best validation loss

This is useful when we want to continue training later.


In [ ]:
full_checkpoint = {
    "epoch": improved_best_epoch,
    "model_state_dict": improved_model.state_dict(),
    "optimizer_state_dict": improved_optimizer.state_dict(),
    "best_val_loss": improved_best_val_loss,
}

torch.save(full_checkpoint, "full_training_checkpoint.pt")
print("Full checkpoint saved.")


## Part Q — Dropout vs Weight Decay vs Early Stopping

### Dropout
- randomly removes some neurons during training
- reduces dependence on specific neurons
- good for regularization

### Weight Decay
- penalizes very large weights
- keeps the model simpler
- often improves generalization

### Early Stopping
- stops training when validation no longer improves
- prevents useless extra epochs
- usually works best with checkpoint saving


## Part R — A Small Dropout Experiment

Let us quickly compare different dropout values.


In [ ]:
dropout_results = []

for rate in [0.0, 0.2, 0.5]:
    model = DropoutNet(input_size, dropout_rate=rate).to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

    history, best_loss, best_epoch = train_model(
        model,
        train_loader,
        val_loader,
        nn.BCEWithLogitsLoss(),
        optimizer,
        epochs=20,
        patience=5,
        checkpoint_path=None,
    )

    test_acc, _, _ = evaluate_model(model, test_loader)

    dropout_results.append({
        "Dropout Rate": rate,
        "Best Validation Loss": best_loss,
        "Best Validation Epoch": best_epoch,
        "Test Accuracy": test_acc,
    })

pd.DataFrame(dropout_results).round(4)


### What students should learn here

There is no guarantee that one dropout value always wins.

Model training depends on:
- data
- architecture
- learning rate
- regularization strength
- number of epochs

That is why we experiment and compare.


## Part S — Common Beginner Mistakes

1. using only training accuracy and ignoring validation loss  
2. training for too many epochs without checking validation performance  
3. forgetting to use `model.eval()` during evaluation  
4. fitting scaler on the whole dataset  
5. loading the wrong checkpoint file  
6. saving only the last model instead of the best model  
7. thinking high training accuracy always means a good model


## Part T — Mini In-Class Practice

Do these in class:

1. Change dropout rate from `0.3` to `0.5` and retrain  
2. Change `weight_decay` from `1e-3` to `1e-4` and compare  
3. Reduce patience from `10` to `5` and observe when training stops  
4. Train for only `20` epochs without early stopping  
5. Compare baseline and improved model test accuracy


## Part U — 10 After-Lab Tasks

### Task 1
Load any binary classification dataset from scikit-learn and split it into train, validation, and test sets.

### Task 2
Scale the input data using `StandardScaler` and convert it into PyTorch tensors.

### Task 3
Create a neural network without dropout and train it. Plot train loss and validation loss.

### Task 4
Write two lines explaining whether the baseline model looks underfit, well-fit, or overfit.

### Task 5
Add `Dropout` layers in the model and train again. Compare results with the baseline.

### Task 6
Train a model using `weight_decay` in the optimizer and compare its validation loss with the baseline.

### Task 7
Use early stopping with patience `5` or `10` and save the best checkpoint.

### Task 8
Load the saved checkpoint again and calculate test accuracy.

### Task 9
Create a small result table with:
- model name
- best validation loss
- best epoch
- test accuracy

### Task 10
Write a short paragraph explaining:
- what overfitting is
- why validation data is important
- when dropout, weight decay, and early stopping are useful


## Part V — Optional Homework Challenge

Use a different classification dataset from scikit-learn.

Do the following:

1. prepare train, validation, and test data  
2. scale the features correctly  
3. build a baseline network  
4. build a dropout network  
5. build a weight decay version  
6. apply early stopping  
7. save the best checkpoint  
8. load the checkpoint again  
9. compare all models in a result table  
10. write a short reflection on which method helped most and why


## Part W — Final Revision Notes

Today you learned:

- training does not only mean running epochs
- we must monitor both training and validation results
- overfitting happens when the model learns training details too strongly
- dropout is a regularization method
- weight decay is another regularization method
- early stopping can stop useless extra training
- saving the best checkpoint is very important
- validation loss is often one of the most useful signals during training
- good deep learning practice means train, monitor, compare, and save carefully
